<a href="https://colab.research.google.com/github/010Ankushsharma/text_classification_ml_notebook/blob/main/text_classification_ml_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Text Classification: End-to-End Machine Learning Pipeline

---

## Introduction

This notebook provides a **complete, modular, and production-ready pipeline** for text classification using a large-scale dataset (~1.2 million rows). It is designed to be **flexible** — you can mix and match any vectorization method with any model.

### 📌 Dataset
- **`sentence`** → Raw text data
- **`tags`** → Target labels (multi-class classification)

### 🗺️ Notebook Structure

| # | Section | Description |
|---|---------|-------------|
| 1 | Setup & Imports | Install and import all dependencies |
| 2 | Data Loading | Load dataset efficiently |
| 3 | EDA | Explore distributions and patterns |
| 4 | Preprocessing | Clean and normalize text |
| 5 | Vectorization | TF-IDF / Word2Vec / GloVe / Sentence Transformers |
| 6 | Model Building | LR / RF / XGBoost / LSTM / BERT |
| 7 | Evaluation | Metrics + Confusion Matrix |
| 8 | Comparison | Compare all combinations |
| 9 | Optimization | Hyperparameter tuning + Model saving |

### ⚡ Flexibility Note
> **You can run any vectorization cell independently, then plug the resulting features into any model cell. Each section is self-contained and modular.**

---

## Section 1 — ⚙️ Setup & Dependencies

Install all required libraries. Run this cell once. GPU-enabled TensorFlow and PyTorch will automatically use CUDA if available.

In [ ]:
# ─────────────────────────────────────────────────
# Install Dependencies
# ─────────────────────────────────────────────────
# Uncomment and run once to install all required packages

# !pip install pandas numpy scikit-learn matplotlib seaborn
# !pip install gensim spacy nltk
# !pip install xgboost lightgbm
# !pip install tensorflow torch torchvision
# !pip install sentence-transformers transformers
# !pip install optuna joblib tqdm
# !pip install dask[dataframe]
# !python -m spacy download en_core_web_sm

# Download NLTK resources
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Setup complete.")

In [ ]:
# ─────────────────────────────────────────────────
# Core Imports
# ─────────────────────────────────────────────────
import os
import re
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

# NLP
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from scipy.sparse import issparse

# XGBoost / LightGBM
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("⚠️  XGBoost not found — using LightGBM fallback.")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False

# Gensim (Word2Vec)
from gensim.models import Word2Vec, KeyedVectors

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# TensorFlow / Keras
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import (
        Embedding, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
    )
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    TF_AVAILABLE = True
    print(f"✅ TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU')}")
except ImportError:
    TF_AVAILABLE = False
    print("⚠️  TensorFlow not available.")

# HuggingFace Transformers (BERT)
try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer
    )
    import torch
    from torch.utils.data import Dataset, DataLoader
    TRANSFORMERS_AVAILABLE = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"✅ Transformers available | Device: {DEVICE}")
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("⚠️  HuggingFace Transformers not available.")

# ─── Global Configuration ───────────────────────
RANDOM_STATE   = 42
TEST_SIZE      = 0.20         # 80/20 split
MAX_FEATURES   = 50_000       # for TF-IDF / Keras Tokenizer
MAX_SEQ_LEN    = 128          # max token length for LSTM / BERT
BATCH_SIZE     = 512          # training batch size
SAMPLE_SIZE    = 200_000      # reduce for quick experiments; set None to use full dataset
RESULTS        = {}           # store evaluation results for comparison

np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted')
tqdm.pandas()

print("\n✅ All imports successful.")

---
## Section 2 — 📂 Data Loading

Load the dataset efficiently. For 1.2M rows we use chunked reading when memory is limited, and optional Dask support for very large files.

> **Tip:** Set `SAMPLE_SIZE` above to a smaller number (e.g. `50_000`) for rapid prototyping.

In [ ]:
# ─────────────────────────────────────────────────
# Data Loading
# ─────────────────────────────────────────────────

DATA_PATH = 'your_dataset.csv'          # ← Update this path

def load_data(path, sample_size=None, use_dask=False, chunksize=100_000):
    """
    Load CSV data with optional sampling and chunked reading.

    Parameters
    ----------
    path        : str   — file path
    sample_size : int   — rows to sample (None = all)
    use_dask    : bool  — use Dask for out-of-core loading
    chunksize   : int   — rows per chunk (used only with chunked mode)
    """
    if use_dask:
        import dask.dataframe as dd
        ddf = dd.read_csv(path, usecols=['sentence', 'tags'])
        df  = ddf.compute()
        print(f"✅ Loaded via Dask: {len(df):,} rows")
    elif sample_size:
        # Efficient: skip rows probabilistically without loading all data
        total = sum(1 for _ in open(path)) - 1  # exclude header
        skip  = sorted(np.random.choice(
            range(1, total + 1),
            total - sample_size,
            replace=False
        ))
        df = pd.read_csv(path, skiprows=skip, usecols=['sentence', 'tags'])
        print(f"✅ Sampled {len(df):,} rows from {total:,} total")
    else:
        # Chunked loading — memory efficient for full dataset
        chunks = []
        for chunk in pd.read_csv(path, usecols=['sentence', 'tags'], chunksize=chunksize):
            chunks.append(chunk)
        df = pd.concat(chunks, ignore_index=True)
        print(f"✅ Full dataset loaded: {len(df):,} rows")
    return df


# ── Load Data ───────────────────────────────────
# Option A — Load with sampling (fast, for experimentation)
df = load_data(DATA_PATH, sample_size=SAMPLE_SIZE)

# Option B — Load full dataset (comment out Option A)
# df = load_data(DATA_PATH, sample_size=None)

# Option C — Dask (for machines with limited RAM)
# df = load_data(DATA_PATH, use_dask=True)

print(f"\nShape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

---
## Section 3 — 🔍 Exploratory Data Analysis (EDA)

Understand the dataset before modelling. Key questions:
- Are the classes balanced?
- What is the typical sentence length?
- Are there missing values?

In [ ]:
# ─────────────────────────────────────────────────
# Basic Statistics
# ─────────────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total rows     : {len(df):,}")
print(f"Missing (sentence) : {df['sentence'].isnull().sum():,}")
print(f"Missing (tags)     : {df['tags'].isnull().sum():,}")
print(f"Unique classes : {df['tags'].nunique()}")
print("\nClass Distribution:")
print(df['tags'].value_counts())
print()
df.info()

In [ ]:
# ─────────────────────────────────────────────────
# Visualisations
# ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Class Distribution (Top 20)
top_classes = df['tags'].value_counts().head(20)
sns.barplot(x=top_classes.values, y=top_classes.index, ax=axes[0], palette='viridis')
axes[0].set_title('Top 20 Class Distributions', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')

# 2. Sentence Length Distribution
df['sentence_len'] = df['sentence'].fillna('').apply(lambda x: len(x.split()))
axes[1].hist(df['sentence_len'].clip(upper=200), bins=60, color='steelblue', edgecolor='white')
axes[1].set_title('Sentence Length Distribution (words)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

# 3. Class Imbalance Ratio
counts = df['tags'].value_counts()
imbalance_ratio = counts.max() / counts.min()
axes[2].pie(
    counts.head(8),
    labels=counts.head(8).index,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('pastel')
)
axes[2].set_title(f'Class Share (Top 8) | Imbalance Ratio: {imbalance_ratio:.1f}x',
                  fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Sentence length stats:")
print(df['sentence_len'].describe().to_string())
print(f"\n⚠️  Imbalance ratio (max/min class): {imbalance_ratio:.1f}x")
if imbalance_ratio > 5:
    print("   → Significant imbalance detected. Consider class_weight='balanced' or oversampling.")

---
## Section 4 — 🧹 Data Cleaning & Preprocessing

Steps applied:
1. Drop nulls
2. Lowercase
3. Remove punctuation & special characters
4. Remove stopwords
5. Tokenize
6. Lemmatize (spaCy)
7. Encode labels
8. Handle class imbalance

In [ ]:
# ─────────────────────────────────────────────────
# spaCy Setup
# ─────────────────────────────────────────────────
# Load spaCy model (disable unused components for speed)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()  # NLTK fallback

print("✅ spaCy model loaded.")

In [ ]:
# ─────────────────────────────────────────────────
# Text Cleaning Function
# ─────────────────────────────────────────────────

def clean_text(text):
    """
    Apply full NLP preprocessing pipeline to a single string.
    Returns a cleaned, lemmatized sentence.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # 4. Remove punctuation & special characters (keep spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # 5. Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def lemmatize_spacy(texts, batch_size=1000, n_process=1):
    """
    Lemmatize + remove stopwords using spaCy pipe.
    Efficient for large corpora via batching.

    Parameters
    ----------
    texts      : list of str
    batch_size : int — spaCy internal batch size
    n_process  : int — parallel workers (use -1 for all cores)
    """
    processed = []
    for doc in tqdm(
        nlp.pipe(texts, batch_size=batch_size, n_process=n_process),
        total=len(texts),
        desc='Lemmatizing'
    ):
        tokens = [
            token.lemma_ for token in doc
            if not token.is_stop
            and not token.is_punct
            and token.lemma_.strip()
            and len(token.lemma_) > 1
        ]
        processed.append(' '.join(tokens))
    return processed


print("✅ Preprocessing functions defined.")

# Quick sanity test
sample = "The quick-brown fox JUMPS over the lazy dog!! Visit https://example.com"
print(f"\nOriginal : {sample}")
print(f"Cleaned  : {clean_text(sample)}")

In [ ]:
# ─────────────────────────────────────────────────
# Apply Preprocessing Pipeline
# ─────────────────────────────────────────────────
print("Step 1/3 — Dropping nulls...")
df = df.dropna(subset=['sentence', 'tags']).reset_index(drop=True)

print("Step 2/3 — Cleaning text...")
df['cleaned'] = df['sentence'].progress_apply(clean_text)

# Remove empty strings after cleaning
df = df[df['cleaned'].str.strip() != ''].reset_index(drop=True)

print("Step 3/3 — Lemmatizing (spaCy)...")
# NOTE: For full 1.2M rows use n_process=-1 for multiprocessing
df['processed'] = lemmatize_spacy(df['cleaned'].tolist(), batch_size=2000, n_process=1)

print(f"\n✅ Preprocessing complete. Rows retained: {len(df):,}")
df[['sentence', 'cleaned', 'processed', 'tags']].head(5)

In [ ]:
# ─────────────────────────────────────────────────
# Label Encoding & Class Imbalance Handling
# ─────────────────────────────────────────────────
le = LabelEncoder()
df['label'] = le.fit_transform(df['tags'])

N_CLASSES = df['label'].nunique()
print(f"Number of classes : {N_CLASSES}")
print(f"Class names       : {le.classes_}")

# ── Compute class weights for imbalanced datasets ──
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df['label']),
    y=df['label']
)
CLASS_WEIGHT_DICT = dict(enumerate(class_weights_array))
print(f"\nClass weights (sample): {dict(list(CLASS_WEIGHT_DICT.items())[:5])}")
print("→ These will be passed to models that support class_weight='balanced'")

In [ ]:
# ─────────────────────────────────────────────────
# Train / Test Split
# ─────────────────────────────────────────────────
X_raw    = df['processed'].values     # preprocessed text (for TF-IDF / Word2Vec)
X_orig   = df['sentence'].values      # original text (for Sentence Transformers / BERT)
y        = df['label'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train_orig, X_test_orig = train_test_split(
    X_orig,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train set : {len(X_train_raw):,}  ({100*(1-TEST_SIZE):.0f}%)")
print(f"Test set  : {len(X_test_raw):,}  ({100*TEST_SIZE:.0f}%)")

---
## Section 5 — 🔢 Feature Engineering (Vectorization)

> **⚡ Modular Design:** Run **any one** of the cells below. The variable `X_train_vec` and `X_test_vec` will hold the features for the chosen method. Then proceed directly to Section 6 — Model Building.

| Method | Best For | Speed | Semantic Capture |
|--------|----------|-------|------------------|
| TF-IDF | Keyword-heavy tasks, baselines | ⚡⚡⚡ | ⭐ |
| Word2Vec | When you have domain corpus | ⚡⚡ | ⭐⭐ |
| GloVe/FastText | General language tasks | ⚡⚡ | ⭐⭐⭐ |
| Sentence Transformers | Semantic similarity, best accuracy | ⚡ | ⭐⭐⭐⭐⭐ |

### 5A — TF-IDF Vectorization

**When to use:** TF-IDF is a strong baseline for any text classification task. It works exceptionally well when the discriminating signal is in *specific words or phrases* (e.g., topic classification, spam detection). It produces sparse matrices — very memory efficient for large datasets. Use unigrams for speed, or `ngram_range=(1,2)` for richer features.

In [ ]:
# ─────────────────────────────────────────────────
# VECTORIZATION METHOD A: TF-IDF
# ─────────────────────────────────────────────────
print("🔢 Fitting TF-IDF Vectorizer...")
t0 = time.time()

tfidf_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=(1, 2),          # unigrams + bigrams
    sublinear_tf=True,           # apply log normalization to TF
    min_df=3,                    # ignore very rare terms
    max_df=0.95,                 # ignore overly common terms
    strip_accents='unicode',
    analyzer='word'
)

X_train_vec = tfidf_vectorizer.fit_transform(X_train_raw)   # sparse matrix
X_test_vec  = tfidf_vectorizer.transform(X_test_raw)

print(f"✅ TF-IDF complete in {time.time()-t0:.1f}s")
print(f"   Feature shape : {X_train_vec.shape}")
print(f"   Matrix type   : {type(X_train_vec)} (sparse — memory efficient ✅)")
print(f"   Top 10 terms  : {tfidf_vectorizer.get_feature_names_out()[:10]}")
VECTORIZER_NAME = 'TF-IDF'

### 5B — Word2Vec (Trained on Corpus)

**When to use:** Word2Vec captures semantic relationships between words. Train it on your own domain corpus for best results. It does **not** capture word order or full sentence semantics. Ideal when your domain is specialised (medical, legal, etc.) and pre-trained embeddings may not fit well.

In [ ]:
# ─────────────────────────────────────────────────
# VECTORIZATION METHOD B: Word2Vec
# ─────────────────────────────────────────────────

def tokens_from_series(series):
    """Tokenize a pandas Series into list of word lists."""
    return [str(s).split() for s in series]


def average_word_vectors(sentences, model, vocab, embedding_dim):
    """
    Represent each sentence as the average of its word vectors.
    Out-of-vocabulary words are skipped.
    """
    features = np.zeros((len(sentences), embedding_dim), dtype='float32')
    for idx, sentence in enumerate(tqdm(sentences, desc='Averaging vectors')):
        words  = str(sentence).split()
        vecs   = [model[w] for w in words if w in vocab]
        if vecs:
            features[idx] = np.mean(vecs, axis=0)
    return features


print("🔢 Training Word2Vec model...")
t0 = time.time()

W2V_DIM     = 200
W2V_WINDOW  = 5
W2V_WORKERS = 4
W2V_EPOCHS  = 10

train_tokens = tokens_from_series(X_train_raw)

w2v_model = Word2Vec(
    sentences  = train_tokens,
    vector_size= W2V_DIM,
    window     = W2V_WINDOW,
    min_count  = 3,
    workers    = W2V_WORKERS,
    epochs     = W2V_EPOCHS,
    sg         = 0           # CBOW (faster); sg=1 for Skip-gram
)

w2v_vocab = set(w2v_model.wv.key_to_index.keys())
print(f"✅ Word2Vec trained in {time.time()-t0:.1f}s | Vocab size: {len(w2v_vocab):,}")

# Save model for reuse
w2v_model.save('word2vec_model.bin')

print("\nAveraging word vectors for sentences...")
X_train_vec = average_word_vectors(X_train_raw, w2v_model.wv, w2v_vocab, W2V_DIM)
X_test_vec  = average_word_vectors(X_test_raw,  w2v_model.wv, w2v_vocab, W2V_DIM)

print(f"\n✅ Word2Vec features ready.")
print(f"   Shape: {X_train_vec.shape}")
VECTORIZER_NAME = 'Word2Vec'

### 5C — Pretrained Embeddings (GloVe / FastText)

**When to use:** Pre-trained embeddings are trained on massive corpora (Wikipedia, Common Crawl) and encode broad linguistic knowledge. Use them when your dataset is small or medium-sized. FastText also handles out-of-vocabulary words via character n-grams — great for noisy or morphologically rich text.

> **Download links:**
> - GloVe: https://nlp.stanford.edu/projects/glove/ (`glove.6B.zip`)
> - FastText: https://fasttext.cc/docs/en/english-vectors.html

In [ ]:
# ─────────────────────────────────────────────────
# VECTORIZATION METHOD C: GloVe / FastText Pretrained
# ─────────────────────────────────────────────────

GLOVE_PATH   = 'glove.6B.200d.txt'      # ← Update path
# FASTTEXT_PATH = 'cc.en.300.bin'        # FastText binary (uncomment if preferred)
EMBED_DIM    = 200


def load_glove_embeddings(filepath, embedding_dim):
    """Load GloVe text file into a word → vector dict."""
    embeddings = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc='Loading GloVe'):
            parts = line.split()
            word  = parts[0]
            vec   = np.array(parts[1:], dtype='float32')
            if len(vec) == embedding_dim:
                embeddings[word] = vec
    return embeddings


def load_fasttext_embeddings(filepath):
    """Load FastText .bin model (handles OOV via subwords)."""
    from gensim.models.fasttext import load_facebook_model
    return load_facebook_model(filepath).wv


def sentence_to_embedding(sentence, embedding_dict, embedding_dim):
    """Average word embeddings for a sentence."""
    words = str(sentence).split()
    vecs  = [embedding_dict[w] for w in words if w in embedding_dict]
    if vecs:
        return np.mean(vecs, axis=0)
    return np.zeros(embedding_dim)


print("🔢 Loading GloVe embeddings...")
t0 = time.time()

glove_embeddings = load_glove_embeddings(GLOVE_PATH, EMBED_DIM)
print(f"✅ Loaded {len(glove_embeddings):,} GloVe vectors in {time.time()-t0:.1f}s")

print("\nBuilding sentence vectors...")
X_train_vec = np.vstack([
    sentence_to_embedding(s, glove_embeddings, EMBED_DIM)
    for s in tqdm(X_train_raw, desc='Train')
])
X_test_vec = np.vstack([
    sentence_to_embedding(s, glove_embeddings, EMBED_DIM)
    for s in tqdm(X_test_raw, desc='Test')
])

print(f"\n✅ GloVe features ready. Shape: {X_train_vec.shape}")
VECTORIZER_NAME = 'GloVe'

### 5D — Sentence Transformers (Best Semantic Embeddings)

**When to use:** Sentence Transformers produce contextual, fixed-size embeddings for full sentences — not just words. They capture meaning, synonyms, and paraphrases. This is the **strongest baseline** for tasks requiring semantic understanding (intent detection, paraphrase detection, NLI).

**Limitation:** Slower than TF-IDF/Word2Vec. For 1.2M rows, GPU is strongly recommended. Use batching to avoid memory errors.

Model options:
- `all-MiniLM-L6-v2` — fast, good quality (recommended default)
- `all-mpnet-base-v2` — highest quality, slower
- `paraphrase-multilingual-MiniLM-L12-v2` — multilingual

In [ ]:
# ─────────────────────────────────────────────────
# VECTORIZATION METHOD D: Sentence Transformers
# ─────────────────────────────────────────────────
ST_MODEL_NAME = 'all-MiniLM-L6-v2'    # swap for higher-quality model if needed
ST_BATCH_SIZE = 256 if DEVICE == 'cuda' else 64

print(f"🔢 Loading Sentence Transformer: {ST_MODEL_NAME}")
print(f"   Device: {DEVICE}")
t0 = time.time()

st_model = SentenceTransformer(ST_MODEL_NAME, device=DEVICE)

print("\nEncoding training set (batched)...")
X_train_vec = st_model.encode(
    X_train_orig.tolist(),
    batch_size=ST_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("\nEncoding test set (batched)...")
X_test_vec = st_model.encode(
    X_test_orig.tolist(),
    batch_size=ST_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Sentence Transformer embeddings ready in {time.time()-t0:.1f}s")
print(f"   Shape: {X_train_vec.shape}  (each sentence → {X_train_vec.shape[1]}-dim vector)")

# Optional: Save embeddings to disk to avoid re-encoding
np.save('X_train_st.npy', X_train_vec)
np.save('X_test_st.npy',  X_test_vec)
print("   Saved to X_train_st.npy / X_test_st.npy")

VECTORIZER_NAME = 'SentenceTransformer'

---
## Section 6 — 🤖 Model Building

> **⚡ Modular Design:** Before running any model cell, make sure `X_train_vec`, `X_test_vec`, `y_train`, `y_test` are set from **Section 5**. Each cell below is independently runnable.

The helper function `evaluate_model()` is defined first and shared across all models.

In [ ]:
# ─────────────────────────────────────────────────
# Shared Evaluation Helper
# ─────────────────────────────────────────────────

def evaluate_model(model_name, y_true, y_pred, y_prob=None, fit_time=None):
    """
    Compute and display classification metrics.
    Results are stored in the global RESULTS dict for comparison.

    Parameters
    ----------
    model_name : str
    y_true     : array-like
    y_pred     : array-like
    y_prob     : array-like (optional, for ROC-AUC)
    fit_time   : float (seconds, optional)
    """
    avg = 'weighted' if N_CLASSES > 2 else 'binary'

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=avg, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=avg, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=avg, zero_division=0)

    key = f"{VECTORIZER_NAME} + {model_name}"
    RESULTS[key] = {
        'Accuracy'  : round(acc,  4),
        'Precision' : round(prec, 4),
        'Recall'    : round(rec,  4),
        'F1-Score'  : round(f1,   4),
        'Fit Time'  : round(fit_time, 2) if fit_time else None
    }

    print("\n" + "═" * 55)
    print(f" 📊 {key}")
    print("═" * 55)
    print(f" Accuracy  : {acc:.4f}")
    print(f" Precision : {prec:.4f}")
    print(f" Recall    : {rec:.4f}")
    print(f" F1-Score  : {f1:.4f}")
    if fit_time: print(f" Fit Time  : {fit_time:.2f}s")
    print()
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(6, N_CLASSES), max(5, N_CLASSES - 1)))
    sns.heatmap(
        cm, annot=(N_CLASSES <= 15), fmt='d',
        xticklabels=le.classes_, yticklabels=le.classes_,
        cmap='Blues'
    )
    plt.title(f'Confusion Matrix — {key}', fontsize=12, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name.replace(" ", "_")}.png', dpi=120)
    plt.show()

print("✅ evaluate_model() defined.")

### Model 1 — Logistic Regression

In [ ]:
# ─────────────────────────────────────────────────
# MODEL 1: Logistic Regression
# ─────────────────────────────────────────────────
# Strengths : Fast, interpretable, strong baseline
# Works well with TF-IDF (sparse features)
# ─────────────────────────────────────────────────
print("Training Logistic Regression...")
t0 = time.time()

lr_model = LogisticRegression(
    max_iter    = 1000,
    C           = 1.0,
    solver      = 'saga',        # fast for large datasets
    class_weight= 'balanced',    # handles imbalance
    n_jobs      = -1,
    random_state= RANDOM_STATE
)

lr_model.fit(X_train_vec, y_train)
fit_time = time.time() - t0

y_pred_lr = lr_model.predict(X_test_vec)
evaluate_model('Logistic Regression', y_test, y_pred_lr, fit_time=fit_time)

# Save model
joblib.dump(lr_model, 'model_lr.pkl')
print("✅ Model saved → model_lr.pkl")

### Model 2 — Random Forest

In [ ]:
# ─────────────────────────────────────────────────
# MODEL 2: Random Forest
# ─────────────────────────────────────────────────
# Strengths : Robust to noise, non-linear, feature importance
# Note      : Memory-heavy on sparse matrices — convert if needed
# ─────────────────────────────────────────────────
print("Training Random Forest...")
t0 = time.time()

# Convert sparse to dense only if matrix is small enough
X_tr = X_train_vec.toarray() if issparse(X_train_vec) and X_train_vec.shape[1] < 10_000 else X_train_vec
X_te = X_test_vec.toarray()  if issparse(X_test_vec)  and X_test_vec.shape[1]  < 10_000 else X_test_vec

rf_model = RandomForestClassifier(
    n_estimators  = 200,
    max_depth     = 30,
    class_weight  = 'balanced_subsample',
    n_jobs        = -1,
    random_state  = RANDOM_STATE,
    min_samples_leaf = 2
)

rf_model.fit(X_tr, y_train)
fit_time = time.time() - t0

y_pred_rf = rf_model.predict(X_te)
evaluate_model('Random Forest', y_test, y_pred_rf, fit_time=fit_time)

joblib.dump(rf_model, 'model_rf.pkl')
print("✅ Model saved → model_rf.pkl")

### Model 3 — XGBoost / LightGBM

In [ ]:
# ─────────────────────────────────────────────────
# MODEL 3: XGBoost (falls back to LightGBM)
# ─────────────────────────────────────────────────
# Strengths  : Gradient boosting — often best ML model
# LightGBM   : Faster on large datasets, lower memory
# XGBoost    : Better with dense numeric features
# ─────────────────────────────────────────────────
print("Training Gradient Boosting model...")
t0 = time.time()

# Convert sparse to dense if using XGBoost (it prefers DMatrix)
if issparse(X_train_vec):
    X_tr_xgb = X_train_vec  # XGBoost handles sparse natively
    X_te_xgb = X_test_vec
else:
    X_tr_xgb = X_train_vec
    X_te_xgb = X_test_vec

if XGB_AVAILABLE:
    xgb_model = xgb.XGBClassifier(
        n_estimators     = 300,
        max_depth        = 6,
        learning_rate    = 0.1,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        use_label_encoder= False,
        eval_metric      = 'mlogloss',
        tree_method      = 'hist',   # GPU: 'gpu_hist'
        n_jobs           = -1,
        random_state     = RANDOM_STATE
    )
    xgb_model.fit(
        X_tr_xgb, y_train,
        eval_set=[(X_te_xgb, y_test)],
        verbose=100
    )
    y_pred_xgb = xgb_model.predict(X_te_xgb)
    evaluate_model('XGBoost', y_test, y_pred_xgb, fit_time=time.time()-t0)
    joblib.dump(xgb_model, 'model_xgb.pkl')
    print("✅ Saved → model_xgb.pkl")

elif LGB_AVAILABLE:
    print("  XGBoost not found — using LightGBM")
    lgb_model = lgb.LGBMClassifier(
        n_estimators   = 300,
        learning_rate  = 0.1,
        num_leaves     = 63,
        class_weight   = 'balanced',
        n_jobs         = -1,
        random_state   = RANDOM_STATE
    )
    lgb_model.fit(X_tr_xgb, y_train)
    y_pred_xgb = lgb_model.predict(X_te_xgb)
    evaluate_model('LightGBM', y_test, y_pred_xgb, fit_time=time.time()-t0)
    joblib.dump(lgb_model, 'model_lgb.pkl')
    print("✅ Saved → model_lgb.pkl")

else:
    print("⚠️  Neither XGBoost nor LightGBM available. Please install one.")

### Model 4 — BiLSTM (TensorFlow/Keras)

Deep learning model for sequential text understanding. BiLSTM reads sentences in both directions, capturing forward and backward context.

> **Requirements:** TensorFlow + GPU recommended for training speed.

In [ ]:
# ─────────────────────────────────────────────────
# MODEL 4: BiLSTM (TensorFlow/Keras)
# ─────────────────────────────────────────────────
if not TF_AVAILABLE:
    print("⚠️  TensorFlow not available. Skipping BiLSTM.")
else:
    # ── Tokenization ──────────────────────────────
    print("Tokenizing for LSTM...")
    keras_tokenizer = Tokenizer(num_words=MAX_FEATURES, oov_token='<OOV>')
    keras_tokenizer.fit_on_texts(X_train_raw)

    X_train_seq = keras_tokenizer.texts_to_sequences(X_train_raw)
    X_test_seq  = keras_tokenizer.texts_to_sequences(X_test_raw)

    X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
    X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

    print(f"  Padded shape: {X_train_pad.shape}")

    # ── Model Architecture ───────────────────────
    EMBED_DIM_LSTM = 128
    LSTM_UNITS     = 128

    lstm_model = Sequential([
        Embedding(
            input_dim    = MAX_FEATURES + 1,
            output_dim   = EMBED_DIM_LSTM,
            input_length = MAX_SEQ_LEN,
            mask_zero    = True
        ),
        Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(LSTM_UNITS // 2)),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(N_CLASSES, activation='softmax')
    ])

    lstm_model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy']
    )
    lstm_model.summary()

    # ── Training ─────────────────────────────────
    callbacks = [
        EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_bilstm.h5', save_best_only=True, verbose=0)
    ]

    print("\nTraining BiLSTM...")
    t0 = time.time()
    history = lstm_model.fit(
        X_train_pad, y_train,
        validation_split = 0.1,
        epochs           = 15,
        batch_size       = BATCH_SIZE,
        class_weight     = CLASS_WEIGHT_DICT,
        callbacks        = callbacks,
        verbose          = 1
    )
    fit_time = time.time() - t0

    # ── Evaluate ──────────────────────────────────
    y_pred_lstm = np.argmax(lstm_model.predict(X_test_pad, batch_size=BATCH_SIZE), axis=1)
    evaluate_model('BiLSTM', y_test, y_pred_lstm, fit_time=fit_time)

    # Plot learning curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('BiLSTM — Loss Curve')
    axes[0].legend()
    axes[1].plot(history.history['accuracy'], label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_title('BiLSTM — Accuracy Curve')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig('bilstm_learning_curves.png', dpi=120)
    plt.show()

    print("✅ BiLSTM saved → best_bilstm.h5")

### Model 5 — BERT Fine-Tuning (HuggingFace Transformers)

State-of-the-art transformer for sequence classification. Fine-tuning BERT gives the best results on most NLP tasks.

> ⚠️ **GPU Required** for reasonable training speed. For 1.2M rows, consider training on a subset or using `distilbert-base-uncased` for efficiency.

In [ ]:
# ─────────────────────────────────────────────────
# MODEL 5: BERT Fine-Tuning (Optional)
# ─────────────────────────────────────────────────
if not TRANSFORMERS_AVAILABLE:
    print("⚠️  HuggingFace Transformers not available. Skipping BERT.")
else:
    BERT_MODEL_NAME = 'distilbert-base-uncased'   # lighter alternative to bert-base
    # BERT_MODEL_NAME = 'bert-base-uncased'         # full BERT
    BERT_MAX_LEN  = 128
    BERT_EPOCHS   = 3
    BERT_LR       = 2e-5
    BERT_BATCH    = 32 if DEVICE == 'cuda' else 8

    # ── Custom Dataset ─────────────────────────────
    class TextDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len):
            self.texts     = list(texts)
            self.labels    = list(labels)
            self.tokenizer = tokenizer
            self.max_len   = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx],
                max_length     = self.max_len,
                padding        = 'max_length',
                truncation     = True,
                return_tensors = 'pt'
            )
            return {
                'input_ids'      : enc['input_ids'].squeeze(),
                'attention_mask' : enc['attention_mask'].squeeze(),
                'labels'         : torch.tensor(self.labels[idx], dtype=torch.long)
            }

    # ── Load Model & Tokenizer ─────────────────────
    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    bert_model     = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME,
        num_labels=N_CLASSES
    ).to(DEVICE)

    # ── Datasets ───────────────────────────────────
    # For large datasets, train on a subset first
    BERT_TRAIN_LIMIT = min(50_000, len(X_train_orig))
    train_ds = TextDataset(X_train_orig[:BERT_TRAIN_LIMIT], y_train[:BERT_TRAIN_LIMIT],
                           bert_tokenizer, BERT_MAX_LEN)
    test_ds  = TextDataset(X_test_orig, y_test, bert_tokenizer, BERT_MAX_LEN)

    # ── Training Arguments ─────────────────────────
    training_args = TrainingArguments(
        output_dir          = './bert_output',
        num_train_epochs    = BERT_EPOCHS,
        per_device_train_batch_size = BERT_BATCH,
        per_device_eval_batch_size  = BERT_BATCH * 2,
        learning_rate       = BERT_LR,
        weight_decay        = 0.01,
        evaluation_strategy = 'epoch',
        save_strategy       = 'epoch',
        load_best_model_at_end = True,
        logging_dir         = './bert_logs',
        logging_steps       = 200,
        fp16                = (DEVICE == 'cuda'),  # mixed precision on GPU
        report_to           = 'none'
    )

    trainer = Trainer(
        model         = bert_model,
        args          = training_args,
        train_dataset = train_ds,
        eval_dataset  = test_ds
    )

    print(f"Fine-tuning {BERT_MODEL_NAME} on {BERT_TRAIN_LIMIT:,} samples...")
    t0 = time.time()
    trainer.train()
    fit_time = time.time() - t0

    # ── Predict ────────────────────────────────────
    preds_output = trainer.predict(test_ds)
    y_pred_bert  = np.argmax(preds_output.predictions, axis=1)
    evaluate_model('BERT (DistilBERT)', y_test, y_pred_bert, fit_time=fit_time)

    # Save
    trainer.save_model('./bert_saved_model')
    print("✅ BERT model saved → ./bert_saved_model")

---
## Section 7 — 📊 Evaluation Summary

Aggregate all results and create a comparison table.

In [ ]:
# ─────────────────────────────────────────────────
# Results Comparison Table
# ─────────────────────────────────────────────────
if not RESULTS:
    print("⚠️  No results found. Run at least one vectorization + model cell.")
else:
    results_df = pd.DataFrame(RESULTS).T.reset_index()
    results_df.columns = ['Vectorizer + Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Fit Time (s)']
    results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

    print("\n" + "═" * 80)
    print("                    MODEL COMPARISON TABLE")
    print("═" * 80)
    print(results_df.to_string(index=False))
    print("═" * 80)

    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    # F1 Score bar chart
    sns.barplot(
        data   = results_df,
        y      = 'Vectorizer + Model',
        x      = 'F1-Score',
        ax     = axes[0],
        palette= 'viridis'
    )
    axes[0].set_title('F1-Score Comparison', fontsize=13, fontweight='bold')
    axes[0].set_xlim(0, 1)
    for bar in axes[0].patches:
        axes[0].text(
            bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.4f}", va='center', fontsize=9
        )

    # Accuracy vs F1 scatter
    scatter = axes[1].scatter(
        results_df['Accuracy'],
        results_df['F1-Score'],
        s   = 150,
        c   = range(len(results_df)),
        cmap= 'plasma'
    )
    for _, row in results_df.iterrows():
        axes[1].annotate(
            row['Vectorizer + Model'],
            (row['Accuracy'], row['F1-Score']),
            textcoords='offset points', xytext=(5, 5), fontsize=8
        )
    axes[1].set_xlabel('Accuracy')
    axes[1].set_ylabel('F1-Score')
    axes[1].set_title('Accuracy vs F1-Score', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    best = results_df.iloc[0]
    print(f"\n🏆 Best Combination : {best['Vectorizer + Model']}")
    print(f"   F1-Score         : {best['F1-Score']}")
    print(f"   Accuracy         : {best['Accuracy']}")

---
## Section 8 — 🔧 Performance Optimization

Strategies for handling 1.2M+ rows efficiently.

In [ ]:
# ─────────────────────────────────────────────────
# Performance Optimization Tips & Code
# ─────────────────────────────────────────────────

# TIP 1: Use sparse matrices
# TF-IDF already returns scipy.sparse — never convert to dense unless forced.
# from scipy.sparse import save_npz, load_npz
# save_npz('X_train_tfidf.npz', X_train_vec)
# X_train_vec = load_npz('X_train_tfidf.npz')   # reload without recomputing


# TIP 2: Mini-batch inference (avoid OOM on large test sets)
def batch_predict(model, X, batch_size=10_000):
    """Run inference in batches to avoid memory overflow."""
    preds = []
    for i in range(0, len(X), batch_size):
        batch = X[i: i + batch_size]
        preds.append(model.predict(batch))
    return np.concatenate(preds)


# TIP 3: Dask for large CSV loading
# import dask.dataframe as dd
# ddf = dd.read_csv('large_file.csv')
# df  = ddf[ddf['sentence'].notnull()].compute()


# TIP 4: Reduce TF-IDF dimensionality with SVD (LSA)
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

def build_lsa_pipeline(n_components=200):
    """
    LSA (Latent Semantic Analysis) = TF-IDF + SVD.
    Reduces sparse TF-IDF to dense low-dimensional representation.
    Useful as input to models that don't work well with sparse matrices.
    """
    return make_pipeline(
        TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2), sublinear_tf=True),
        TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE),
        Normalizer(copy=False)
    )

# lsa_pipeline = build_lsa_pipeline()
# X_train_lsa  = lsa_pipeline.fit_transform(X_train_raw)
# X_test_lsa   = lsa_pipeline.transform(X_test_raw)


# TIP 5: GPU for deep learning (TF auto-detects)
# For PyTorch:
# model = model.to('cuda')
# inputs = inputs.to('cuda')


# TIP 6: Stratified k-fold for large datasets
from sklearn.model_selection import StratifiedKFold
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# for fold, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y)):
#     ...  train on tr_idx, validate on val_idx


print("✅ Optimization utilities defined.")
print("""
📋 OPTIMIZATION CHECKLIST FOR 1.2M ROWS:
─────────────────────────────────────────────────────────
 ☑ Use TF-IDF sparse matrices (never .toarray() unless needed)
 ☑ Set n_process=-1 in spaCy for multi-core lemmatization
 ☑ Use sampling for experimentation; full data only for final run
 ☑ Save TF-IDF / embeddings to disk; skip recomputing on reruns
 ☑ Use GPU for BiLSTM / BERT (10–30x speedup)
 ☑ Use LightGBM over XGBoost for large sparse features
 ☑ For Sentence Transformers: save to .npy; reload with np.load()
 ☑ Use Dask for loading datasets larger than available RAM
 ☑ Set class_weight='balanced' to handle imbalance without oversampling
─────────────────────────────────────────────────────────
""")

---
## Section 9 — 🎛️ Hyperparameter Tuning

Two options: **GridSearchCV** (exhaustive) and **Optuna** (efficient Bayesian optimization, recommended for large search spaces).

In [ ]:
# ─────────────────────────────────────────────────
# Option A: GridSearchCV (Logistic Regression + TF-IDF Pipeline)
# ─────────────────────────────────────────────────
# Build an end-to-end sklearn Pipeline
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=MAX_FEATURES, sublinear_tf=True)),
    ('clf',   LogisticRegression(solver='saga', max_iter=500,
                                  class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [30_000, 50_000],
    'clf__C': [0.1, 1.0, 10.0]
}

print("Running GridSearchCV (Logistic Regression + TF-IDF)...")
print("⚠️  This may take several minutes. Reduce param_grid for speed.")

# Use a subset of training data for faster grid search
GRID_SAMPLE = min(50_000, len(X_train_raw))

grid_search = GridSearchCV(
    lr_pipeline,
    param_grid,
    cv            = 3,
    scoring       = 'f1_weighted',
    n_jobs        = -1,
    verbose       = 2
)
grid_search.fit(X_train_raw[:GRID_SAMPLE], y_train[:GRID_SAMPLE])

print(f"\n✅ Best params : {grid_search.best_params_}")
print(f"   Best F1     : {grid_search.best_score_:.4f}")

# Final evaluation with best pipeline
y_pred_grid = grid_search.predict(X_test_raw)
evaluate_model('LR GridSearch (best)', y_test, y_pred_grid)

joblib.dump(grid_search.best_estimator_, 'model_lr_best_pipeline.pkl')
print("✅ Saved → model_lr_best_pipeline.pkl")

In [ ]:
# ─────────────────────────────────────────────────
# Option B: Optuna (Bayesian Hyperparameter Optimization)
# ─────────────────────────────────────────────────
# Optuna finds good hyperparameters with fewer trials than GridSearch.
# Best suited for XGBoost / LightGBM with many parameters.

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def optuna_objective(trial):
        """Objective function for LightGBM / XGBoost tuning."""
        if LGB_AVAILABLE:
            params = {
                'n_estimators'  : trial.suggest_int('n_estimators', 100, 500),
                'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves'    : trial.suggest_int('num_leaves', 31, 127),
                'max_depth'     : trial.suggest_int('max_depth', 5, 15),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
                'class_weight'  : 'balanced',
                'n_jobs'        : -1,
                'random_state'  : RANDOM_STATE,
                'verbose'       : -1
            }
            model = lgb.LGBMClassifier(**params)
        else:
            params = {
                'n_estimators'  : trial.suggest_int('n_estimators', 100, 400),
                'max_depth'     : trial.suggest_int('max_depth', 4, 10),
                'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample'     : trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'tree_method'   : 'hist',
                'eval_metric'   : 'mlogloss',
                'use_label_encoder': False,
                'n_jobs'        : -1,
                'random_state'  : RANDOM_STATE
            }
            model = xgb.XGBClassifier(**params)

        # Use subset for speed
        N_OPT = min(30_000, len(X_train_vec))
        model.fit(X_train_vec[:N_OPT], y_train[:N_OPT])
        preds = model.predict(X_test_vec)
        return f1_score(y_test, preds, average='weighted', zero_division=0)

    print("Running Optuna optimization (20 trials)...")
    study = optuna.create_study(direction='maximize')
    study.optimize(optuna_objective, n_trials=20, show_progress_bar=True)

    print(f"\n✅ Best Optuna F1 : {study.best_value:.4f}")
    print(f"   Best params    : {study.best_params}")

    # Visualise Optuna results
    try:
        from optuna.visualization.matplotlib import (
            plot_optimization_history, plot_param_importances
        )
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        plot_optimization_history(study, ax=axes[0])
        axes[0].set_title('Optuna — Optimization History')
        plot_param_importances(study, ax=axes[1])
        axes[1].set_title('Optuna — Parameter Importances')
        plt.tight_layout()
        plt.savefig('optuna_results.png', dpi=120)
        plt.show()
    except Exception:
        pass  # Visualisation optional

except ImportError:
    print("⚠️  Optuna not installed. Run: pip install optuna")

---
## Section 10 — 💾 Model Saving & Loading

In [ ]:
# ─────────────────────────────────────────────────
# Model Saving & Loading Reference
# ─────────────────────────────────────────────────

# ── Scikit-learn / XGBoost / LightGBM ───────────
# SAVE
# joblib.dump(lr_model,  'model_lr.pkl')
# joblib.dump(rf_model,  'model_rf.pkl')
# joblib.dump(xgb_model, 'model_xgb.pkl')
# joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
# joblib.dump(le, 'label_encoder.pkl')

# LOAD
# model      = joblib.load('model_lr.pkl')
# vectorizer = joblib.load('tfidf_vectorizer.pkl')
# le         = joblib.load('label_encoder.pkl')
# preds      = model.predict(vectorizer.transform(['new text here']))
# labels     = le.inverse_transform(preds)


# ── TensorFlow / Keras ───────────────────────────
# SAVE
# lstm_model.save('bilstm_model.h5')

# LOAD
# from tensorflow.keras.models import load_model
# lstm_model = load_model('bilstm_model.h5')


# ── PyTorch (BERT) ───────────────────────────────
# SAVE
# torch.save(bert_model.state_dict(), 'bert_model.pt')

# LOAD
# bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=N_CLASSES)
# bert_model.load_state_dict(torch.load('bert_model.pt', map_location=DEVICE))
# bert_model.eval()


# ── Word2Vec ─────────────────────────────────────
# SAVE
# w2v_model.save('word2vec_model.bin')

# LOAD
# w2v_model = Word2Vec.load('word2vec_model.bin')


# ── Production Inference Example ─────────────────
def predict_single(text, vectorizer, model, label_encoder, clean_fn):
    """
    End-to-end prediction for a single new text.

    Parameters
    ----------
    text          : str — raw input
    vectorizer    : fitted TfidfVectorizer
    model         : fitted sklearn classifier
    label_encoder : fitted LabelEncoder
    clean_fn      : preprocessing function
    """
    cleaned  = clean_fn(text)
    features = vectorizer.transform([cleaned])
    pred_idx = model.predict(features)[0]
    proba    = model.predict_proba(features)[0]
    label    = label_encoder.inverse_transform([pred_idx])[0]
    confidence = proba[pred_idx]
    return {'label': label, 'confidence': round(confidence, 4)}

# Example usage (after running TF-IDF + LR):
# result = predict_single("your new sentence here", tfidf_vectorizer, lr_model, le, clean_text)
# print(result)  # → {'label': 'category_name', 'confidence': 0.9341}

print("✅ Saving & loading reference ready.")

---
## Section 11 — 🏁 Comparison & Conclusion

Summary of vectorization + model combinations and practical recommendations.

In [ ]:
# ─────────────────────────────────────────────────
# Final Comparison & Insights
# ─────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════════╗
║          VECTORIZATION × MODEL COMBINATION GUIDE                ║
╠══════════════════════════════════════════════════════════════════╣
║  Use Case          → Recommended Combo                          ║
║  ─────────────────────────────────────────────────────────────  ║
║  Fast baseline     → TF-IDF + Logistic Regression               ║
║  Best ML accuracy  → TF-IDF + XGBoost / LightGBM               ║
║  Semantic tasks    → Sentence Transformers + Logistic Regression ║
║  Domain-specific   → Word2Vec (custom) + Random Forest          ║
║  Best overall      → Sentence Transformers + XGBoost or BERT    ║
║  Low resource      → TF-IDF (sparse) + Logistic Regression      ║
╠══════════════════════════════════════════════════════════════════╣
║  SPEED vs ACCURACY TRADEOFF                                      ║
║  ─────────────────────────────────────────────────────────────  ║
║  Fastest    → TF-IDF + LR         (minutes on 1.2M rows)        ║
║  Balanced   → TF-IDF + LightGBM   (good accuracy, manageable)   ║
║  Best qual. → SentTransformer+LR  (hours, GPU recommended)      ║
║  SOTA       → BERT fine-tuning    (GPU required)                 ║
╠══════════════════════════════════════════════════════════════════╣
║  CLASS IMBALANCE STRATEGIES                                      ║
║  ─────────────────────────────────────────────────────────────  ║
║  1. class_weight='balanced'  (most models)                       ║
║  2. class_weight dict        (Keras LSTM)                        ║
║  3. Stratified splits        (train/test)                        ║
║  4. SMOTE oversampling       (for small datasets only)           ║
╠══════════════════════════════════════════════════════════════════╣
║  SCALING TO 1.2M ROWS                                            ║
║  ─────────────────────────────────────────────────────────────  ║
║  • Sample for experiments; full data for final training          ║
║  • Keep TF-IDF sparse (never convert to dense)                   ║
║  • Use spaCy n_process=-1 for multi-core preprocessing           ║
║  • Cache embeddings to .npy / .npz (avoid re-encoding)          ║
║  • Use Dask for loading if RAM < dataset size                    ║
╚══════════════════════════════════════════════════════════════════╝
""")

if RESULTS:
    results_df = pd.DataFrame(RESULTS).T.reset_index()
    results_df.columns = ['Combo', 'Accuracy', 'Precision', 'Recall', 'F1', 'Time(s)']
    results_df = results_df.sort_values('F1', ascending=False)
    print("\n📊 Your Results Ranked by F1-Score:")
    print(results_df.to_string(index=False))

    best = results_df.iloc[0]
    print(f"\n🏆 Winning Combination: {best['Combo']}")
    print(f"   F1-Score: {best['F1']} | Accuracy: {best['Accuracy']}")

print("\n✅ Notebook complete.")